# Práctica: Búsqueda de Hiperparámetros

**Curso:** Machine Learning — UTEC

---

**Integrantes:**

| # | Nombre completo |
|---|----------------|
| 1 | |
| 2 | |
| 3 | |
| 4 | |

---

## Instrucciones

En esta práctica trabajarán con **3 casos reales** de optimización de hiperparámetros. Para cada caso:

1. Lean el escenario y las pistas proporcionadas
2. **Elijan** cuál de los 3 métodos (Grid Search, Random Search, Bayesian Optimization) es el más apropiado
3. **Justifiquen** su elección en la celda correspondiente
4. **Completen** el código donde se indica con `TODO`
5. **Escriban** sus conclusiones al final de cada caso

**Restricción:** Al finalizar los 3 casos, deben haber utilizado **cada método exactamente una vez**.

| Caso | Método a usar |
|------|---------------|
| 1 | ¿? |
| 2 | ¿? |
| 3 | ¿? |

## Setup

In [3]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 11.5 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.stats import uniform, randint

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


---
## Caso 1: Clasificación de dígitos manuscritos con SVM

**Escenario:** Un servicio postal necesita automatizar la lectura de códigos postales escritos a mano. Se propone usar un clasificador SVM con kernel variable.

**Modelo:** `SVC` (Support Vector Classifier)

**Hiperparámetros a optimizar:**

| Hiperparámetro | Valores candidatos | Cantidad |
|---------------|-------------------|----------|
| `C` | 0.1, 1, 10, 100, 1000 | 5 |
| `kernel` | 'linear', 'rbf', 'poly' | 3 |
| **Total combinaciones** | | **15** |

**Presupuesto:** 15 evaluaciones

### Pistas
- El espacio de búsqueda es **muy pequeño** (solo 15 combinaciones)
- Con 15 evaluaciones, se pueden probar **todas** las combinaciones
- ¿Existe un método que **garantice** encontrar el mejor punto dentro del espacio definido?

### 1.1 Carga y exploración del dataset

In [5]:
from sklearn.datasets import load_digits
from sklearn.svm import SVC

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

# Escalar los datos
scaler_digits = StandardScaler()
X_digits_scaled = scaler_digits.fit_transform(X_digits)

print(f"Dataset: {X_digits.shape[0]} muestras, {X_digits.shape[1]} features")
print(f"Clases: {np.unique(y_digits)} (dígitos 0-9)")
print(f"Distribución balanceada: ~{X_digits.shape[0]//10} muestras por clase")

Dataset: 1797 muestras, 64 features
Clases: [0 1 2 3 4 5 6 7 8 9] (dígitos 0-9)
Distribución balanceada: ~179 muestras por clase


### 1.2 Definición del espacio de hiperparámetros

In [6]:
# Espacio de búsqueda para SVM
param_space_caso1 = {
    'C': [0.1, 1, 10, 100, 1000],
    'kernel': ['linear', 'rbf', 'poly']
}

total_combinaciones = 5 * 3
print(f"Total de combinaciones posibles: {total_combinaciones}")
print(f"Presupuesto disponible: 15 evaluaciones")
print(f"Relación presupuesto/espacio: {15}/{total_combinaciones} = {15/total_combinaciones:.0%}")

Total de combinaciones posibles: 15
Presupuesto disponible: 15 evaluaciones
Relación presupuesto/espacio: 15/15 = 100%


### 1.3 Elección del método

**¿Qué método eligen para este caso?** (Grid Search / Random Search / Bayesian Optimization)

**Método elegido:** `Grid Search`

**Justificación** (escribir al menos 2 razones):

1. Grid es pequeño
2.

### 1.4 Implementación

Completen el código a continuación para implementar el método elegido.

In [7]:
# ============================================
# TODO: Implementar el método elegido
# ============================================
# Instrucciones:
#   1. Crear el objeto de búsqueda con param_space_caso1
#   2. Usar cv=5, scoring='accuracy'
#   3. Ajustar con X_digits_scaled, y_digits
#   4. Guardar resultados en las variables indicadas abajo
#
# Clases útiles de sklearn:
#   - GridSearchCV(estimator, param_grid, cv, scoring, n_jobs)
#   - RandomizedSearchCV(estimator, param_distributions, n_iter, cv, scoring, n_jobs, random_state)
#   - Para Optuna: optuna.create_study() + study.optimize()

modelo_caso1 = SVC()

print("Ejecutando búsqueda de hiperparámetros...")
t0 = time.time()

# --- ESCRIBIR CÓDIGO AQUÍ ---

clf = GridSearchCV(modelo_caso1, param_space_caso1, cv=5, scoring='accuracy')
clf.fit(X_digits_scaled, y_digits)
sorted(clf.cv_results_.keys())


# --- FIN DEL CÓDIGO ---

tiempo_caso1 = time.time() - t0

# TODO: Completar con los resultados de la búsqueda
mejor_score_caso1 = clf.best_score_   # Reemplazar con el mejor score obtenido
mejores_params_caso1 = clf.best_params_  # Reemplazar con los mejores hiperparámetros

print(f"Mejor accuracy: {mejor_score_caso1:.4f}")
print(f"Mejores parámetros: {mejores_params_caso1}")
print(f"Tiempo: {tiempo_caso1:.1f}s")

Ejecutando búsqueda de hiperparámetros...
Mejor accuracy: 0.9538
Mejores parámetros: {'C': 10, 'kernel': 'rbf'}
Tiempo: 6.2s


### 1.5 Conclusiones del Caso 1

**Responder:**

1. ¿El método elegido fue el correcto para este escenario? ¿Por qué?

   *Respuesta: Si, porque el tiempo de ejecución fue relativamente rápido y son pocas combinaciones . Además, se tuvo un accuracy elevado (>95%)> El kernel rbf permite identifcar mejor las relaciones no lineales.

2. ¿Qué pasaría si el espacio de búsqueda tuviera 1000 combinaciones en lugar de 15? ¿Seguirían usando el mismo método?

   *Respuesta:* Sería mucho más intenso computacinalmente, se necesitaría mejor hardware.

---
## Caso 2: Predicción de satisfacción de clientes con Random Forest

**Escenario:** Una empresa de telecomunicaciones quiere predecir qué clientes están insatisfechos para evitar que cancelen su servicio (churn). Se usa un modelo Random Forest con múltiples hiperparámetros.

**Modelo:** `RandomForestClassifier`

**Hiperparámetros a optimizar:**

| Hiperparámetro | Rango | Tipo |
|---------------|-------|------|
| `n_estimators` | [50, 500] | Entero |
| `max_depth` | [3, 30] | Entero |
| `min_samples_split` | [2, 20] | Entero |
| `min_samples_leaf` | [1, 10] | Entero |
| `max_features` | 'sqrt', 'log2', None | Categórico |

**Presupuesto:** 30 evaluaciones

### Pistas
- Hay **5 hiperparámetros** de **tipos mixtos** (enteros + categóricos)
- Un Grid Search exhaustivo necesitaría > 2,000 combinaciones (10 × 10 × 10 × 10 × 3)
- Necesitamos un método que explore bien el espacio con solo 30 evaluaciones
- No necesitamos un modelo sustituto complejo (el modelo no es excesivamente costoso)
- Recuerden el paper de Bergstra & Bengio (2012): ¿qué método cubre más valores únicos por dimensión?

### 2.1 Carga y exploración del dataset

In [8]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

# Simular dataset de churn de telecomunicaciones
X_churn, y_churn = make_classification(
    n_samples=10_000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    weights=[0.75, 0.25],
    random_state=42
)

print(f"Dataset: {X_churn.shape[0]:,} muestras, {X_churn.shape[1]} features")
print(f"Clases: 0=Satisfecho, 1=Insatisfecho")
print(f"Distribución: Satisfecho={np.sum(y_churn==0):,} ({np.mean(y_churn==0)*100:.0f}%), Insatisfecho={np.sum(y_churn==1):,} ({np.mean(y_churn==1)*100:.0f}%)")

Dataset: 10,000 muestras, 20 features
Clases: 0=Satisfecho, 1=Insatisfecho
Distribución: Satisfecho=7,470 (75%), Insatisfecho=2,530 (25%)


### 2.2 Definición del espacio de hiperparámetros

In [9]:
# Espacio de búsqueda para Random Forest
# Para RandomizedSearchCV usar distribuciones:
param_distributions_caso2 = {
    'n_estimators': randint(50, 501),        # [50, 500]
    'max_depth': randint(3, 31),              # [3, 30]
    'min_samples_split': randint(2, 21),      # [2, 20]
    'min_samples_leaf': randint(1, 11),       # [1, 10]
    'max_features': ['sqrt', 'log2', None]
}

# Para GridSearchCV (muy grueso por el presupuesto limitado):
param_grid_caso2 = {
    'n_estimators': [100, 300],
    'max_depth': [5, 15],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 'log2']
}
# 2×2×2×2×2 = 32 combinaciones (ya mayor que 30, y con resolución muy baja)

print(f"Grid Search necesitaría (10 valores por param): {10**4 * 3:,} combinaciones")
print(f"Presupuesto disponible: 30 evaluaciones")
print(f"Relación: 30/{10**4 * 3:,} = {30/(10**4 * 3):.2%} del espacio")

Grid Search necesitaría (10 valores por param): 30,000 combinaciones
Presupuesto disponible: 30 evaluaciones
Relación: 30/30,000 = 0.10% del espacio


### 2.3 Elección del método

**¿Qué método eligen para este caso?** (Grid Search / Random Search / Bayesian Optimization)

**Método elegido:** `random search`

**Justificación** (escribir al menos 2 razones):

1. El número de combinaciones de hiperparámetros es elevado
2.

### 2.4 Implementación

Completen el código a continuación para implementar el método elegido.

In [10]:
# ============================================
# TODO: Implementar el método elegido
# ============================================
# Instrucciones:
#   1. Crear el objeto de búsqueda adecuado
#   2. Usar cv=3, scoring='accuracy', n_iter=30 (si aplica)
#   3. Ajustar con X_churn, y_churn
#   4. Guardar resultados en las variables indicadas
#
# Nota: El diccionario param_distributions_caso2 tiene distribuciones
#       estadísticas (randint, etc.) para muestreo aleatorio.
#       El diccionario param_grid_caso2 tiene listas para grilla.

modelo_caso2 = RandomForestClassifier(random_state=42, n_jobs=-1)

print("Ejecutando búsqueda de hiperparámetros...")
t0 = time.time()

# --- ESCRIBIR CÓDIGO AQUÍ ---
rcv = RandomizedSearchCV(modelo_caso2, param_distributions_caso2, n_iter=30, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)
rcv.fit(X_churn, y_churn)
sorted(rcv.cv_results_.keys())


# --- FIN DEL CÓDIGO ---

tiempo_caso2 = time.time() - t0

# TODO: Completar con los resultados de la búsqueda
mejor_score_caso2 = rcv.best_score_   # Reemplazar con el mejor score obtenido
mejores_params_caso2 = rcv.best_params_  # Reemplazar con los mejores hiperparámetros

print(f"Mejor accuracy: {mejor_score_caso2:.4f}")
print(f"Mejores parámetros: {mejores_params_caso2}")
print(f"Tiempo: {tiempo_caso2:.1f}s")

Ejecutando búsqueda de hiperparámetros...


KeyboardInterrupt: 

### 2.5 Conclusiones del Caso 2

**Responder:**

1. ¿Por qué Grid Search no es práctico para este caso?

   *Respuesta:*  Porque  el número de combinaciones excedería las 30000 lo cúal es muy pesado computacionalmente.

2. ¿Cuántos valores únicos por dimensión explora el método elegido vs. un Grid Search con 30 evaluaciones?

   *Respuesta:* 5 versus 750

3. ¿Qué ventaja tiene usar distribuciones (randint, uniform) en lugar de listas fijas?

   *Respuesta:* Permite buscar combinaciones aleatorias fuera del grid fijo de búsqueda.

---
## Caso 3: Diagnóstico médico con Gradient Boosting

**Escenario:** Un hospital desarrolla un sistema de apoyo al diagnóstico de cáncer de mama. El modelo GradientBoosting es costoso de entrenar (cada evaluación con CV tarda considerablemente). El equipo solo puede permitirse un número muy limitado de evaluaciones.

**Modelo:** `GradientBoostingClassifier`

**Hiperparámetros a optimizar:**

| Hiperparámetro | Rango | Tipo |
|---------------|-------|------|
| `learning_rate` | [0.01, 0.3] | Continuo |
| `n_estimators` | [50, 500] | Entero |
| `max_depth` | [2, 8] | Entero |

**Presupuesto:** 12 evaluaciones (cada una es costosa)

### Pistas
- Cada evaluación es **costosa** (GradientBoosting con CV)
- El presupuesto es **muy limitado** (solo 12 evaluaciones)
- Los hiperparámetros son principalmente **continuos** → espacio suave
- ¿Existe un método que **aprenda** de las evaluaciones anteriores para decidir dónde evaluar a continuación?
- Piensen en el ciclo: modelo sustituto → función de adquisición → nueva evaluación → actualización

### 3.1 Carga y exploración del dataset

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier

cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

# Escalar
scaler_cancer = StandardScaler()
X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)

print(f"Dataset: {X_cancer.shape[0]} muestras, {X_cancer.shape[1]} features")
print(f"Clases: 0=Maligno, 1=Benigno")
print(f"Distribución: Maligno={np.sum(y_cancer==0)} ({np.mean(y_cancer==0)*100:.1f}%), Benigno={np.sum(y_cancer==1)} ({np.mean(y_cancer==1)*100:.1f}%)")

### 3.2 Definición del espacio de hiperparámetros

In [ ]:
# Espacio de búsqueda para GradientBoosting
# Para Optuna:
#   trial.suggest_float('learning_rate', 0.01, 0.3)
#   trial.suggest_int('n_estimators', 50, 500)
#   trial.suggest_int('max_depth', 2, 8)

print("Espacio de búsqueda:")
print(f"  learning_rate: [0.01, 0.3] (continuo)")
print(f"  n_estimators:  [50, 500]   (entero)")
print(f"  max_depth:     [2, 8]      (entero)")
print(f"\nPresupuesto: 12 evaluaciones")
print(f"Nota: Cada evaluación requiere entrenar GradientBoosting + 5-fold CV")

### 3.3 Elección del método

**¿Qué método eligen para este caso?** (Grid Search / Random Search / Bayesian Optimization)

**Método elegido:** `Bayesian optimization`

**Justificación** (escribir al menos 2 razones):

1. Podemos usar resultados anteriores para guiar la búsqueda.
2.

### 3.4 Implementación

Completen el código a continuación para implementar el método elegido.

In [ ]:
# ============================================
# TODO: Implementar el método elegido
# ============================================
# Instrucciones:
#   1. Si usan Optuna, definir una función objective(trial) que:
#      a. Use trial.suggest_float() y trial.suggest_int() para los hiperparámetros
#      b. Cree un GradientBoostingClassifier con esos hiperparámetros
#      c. Calcule cross_val_score con cv=5 y devuelva la media
#   2. Crear el estudio con optuna.create_study(direction='maximize')
#   3. Ejecutar study.optimize(objective, n_trials=12)
#
# Recordatorio de Optuna:
#   study = optuna.create_study(direction='maximize',
#                               sampler=optuna.samplers.TPESampler(seed=42))
#   study.optimize(objective_function, n_trials=N)
#   study.best_value  → mejor score
#   study.best_params → mejores hiperparámetros

print("Ejecutando búsqueda de hiperparámetros...")
t0 = time.time()

# --- ESCRIBIR CÓDIGO AQUÍ ---





# --- FIN DEL CÓDIGO ---

tiempo_caso3 = time.time() - t0

# TODO: Completar con los resultados de la búsqueda
mejor_score_caso3 = None   # Reemplazar con el mejor score obtenido
mejores_params_caso3 = None  # Reemplazar con los mejores hiperparámetros

print(f"Mejor accuracy: {mejor_score_caso3:.4f}")
print(f"Mejores parámetros: {mejores_params_caso3}")
print(f"Tiempo: {tiempo_caso3:.1f}s")

### 3.5 Conclusiones del Caso 3

**Responder:**

1. ¿Cómo aprovecha el método elegido la información de evaluaciones anteriores?

   *Respuesta:*

2. ¿Qué es la función de adquisición (Expected Improvement) y cómo balancea exploración vs. explotación?

   *Respuesta:*

3. ¿Qué diferencia hay entre el TPE (Tree-structured Parzen Estimator) que usa Optuna y un Proceso Gaussiano clásico?

   *Respuesta:*

---
## Resumen comparativo

In [ ]:
# Tabla resumen de los 3 casos
resumen = pd.DataFrame({
    'Caso': [
        '1: Dígitos (SVM)',
        '2: Churn (RandomForest)',
        '3: Cáncer (GradientBoosting)'
    ],
    'Mejor Accuracy': [
        f"{mejor_score_caso1:.4f}",
        f"{mejor_score_caso2:.4f}",
        f"{mejor_score_caso3:.4f}"
    ],
    'Tiempo (s)': [
        f"{tiempo_caso1:.1f}",
        f"{tiempo_caso2:.1f}",
        f"{tiempo_caso3:.1f}"
    ],
    'Método usado': [
        '______',  # TODO: completar
        '______',  # TODO: completar
        '______'   # TODO: completar
    ]
})

print("="*65)
print("RESUMEN DE LA PRÁCTICA")
print("="*65)
print(resumen.to_string(index=False))

## Conclusiones finales

**Completar la siguiente tabla y responder las preguntas:**

| Escenario | Método recomendado | Razón principal |
|-----------|-------------------|------------------|
| Pocos hiperparámetros, espacio pequeño | | |
| Muchos hiperparámetros, tipos mixtos | | |
| Evaluaciones costosas, presupuesto limitado | | |

**Preguntas de reflexión:**

1. ¿En qué caso notaron mayor diferencia entre los métodos? ¿Por qué creen que ocurre?

   *Respuesta:*

2. Si tuvieran que elegir UN solo método para cualquier situación, ¿cuál sería y por qué?

   *Respuesta:*

3. ¿Cómo combinarían los métodos en un flujo de trabajo real? (Pista: exploración → refinamiento)

   *Respuesta:*